In [1]:
import kagglehub
dataset_of_pdf_files_path = kagglehub.dataset_download('manisha717/dataset-of-pdf-files')
print(f"The dataset is loaded to the path: {dataset_of_pdf_files_path}")

/Users/hglabplhak/Documents/Projects/DLMDSDL01/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The dataset is loaded to the path: /Users/hglabplhak/.cache/kagglehub/datasets/manisha717/dataset-of-pdf-files/versions/1


In [9]:
#from AskByRAG import get_dbpath, set_api_env_and_keys
from pathlib import Path
from pypdf import PdfReader
#from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import CharacterTextSplitter
from langchain_aws import BedrockEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
import os


In [3]:
def get_app_key():
    fname = 'app_keyid.sec'
    with open(fname) as f:
        app_key = f.read()
        f.close()
        return app_key

def set_api_env_and_keys():
    app_key = get_app_key()
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_ENDPOINT'] = 'https://withpersona.com/verify?inquiry-id=inq_NMrSJeR6Aiv2XciLNSjc5qcsv2vn'
    os.environ['LANGCHAIN_API_KEY'] = app_key
    os.environ['OPENAI_API_KEY'] = app_key
    return


def get_dbpath():
    home = Path.home()
    db_path = os.path.join(home, 'chroma_vectordb')
    if not os.path.exists(db_path):
        os.makedirs(db_path)
    return db_path

In [10]:
def build_vectors(complete_content):
     # 2. Embed and Store in Vector DB (Chroma)
    splitter = CharacterTextSplitter(
                chunk_size = 2000,
                chunk_overlap = 180,
                )

    chunks = splitter.create_documents(complete_content)

    # 2. Embed and Store in Vector DB (Chroma)
    #embeddings = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0")
    embeddings = OllamaEmbeddings(model="llama3")
    vector_db = Chroma.from_documents(chunks, embedding=embeddings, persist_directory=get_dbpath())
    return vector_db

In [5]:
def extract_text_from_pdf(file_path):
    # creating a pdf reader object
    reader = PdfReader(file_path)
    content = ''
# printing number of pages in pdf file
    page_count = len(reader.pages)
    print(f'Number of pages: {page_count}')
    # getting a specific page from the pdf file
    for index in range(page_count):
        page = reader.pages[index]
        text = page.extract_text()
        content += text
    return content

In [6]:
def read_all_docs(data_paths):
    #absolute_path = data_path + '/Pdf'
    docs_string = ''
    for data_path in data_paths:
        filenames = os.listdir(data_path)
        for filename in filenames:
            file_path = os.path.join(data_path, filename)
            print(f"Processing {file_path}")
            try:
                text = extract_text_from_pdf(file_path)
                docs_string += text
            except Exception as e:
                print(f"Error on {filename}: {e}")
                docs_string += "Failed"
                return -1, docs_string
    return 0, [docs_string]


In [ ]:
#docs_array = read_all_docs(dataset_of_pdf_files_path)
set_api_env_and_keys()
absolute_path = dataset_of_pdf_files_path + '/Pdf'
ret_code, complete_content = read_all_docs(['/Users/hglabplhak/pdfdb', '/Users/hglabplhak/pdfprivdb'])
if ret_code == 0:
    print(complete_content[0])
    print(f"Count of docs: {len(complete_content)}")
    print("build vector")
    vector_db = build_vectors(complete_content)
    print('ready')

Processing /Users/hglabplhak/pdfdb/DieBayerische20231110.pdf
Number of pages: 1
Processing /Users/hglabplhak/pdfdb/JoshuaMDBW_Berichte.pdf
Number of pages: 18
Processing /Users/hglabplhak/pdfdb/The_8086_Micro_Processor_architecture.pdf
Number of pages: 20
Processing /Users/hglabplhak/pdfdb/SC19-6202-05_VM_SP_Operators_Guide_Release_6_Jun88.pdf
Number of pages: 106
Processing /Users/hglabplhak/pdfdb/Revised6_Report_on_the_Algorithmic_Langu.pdf
Number of pages: 70


Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)


Processing /Users/hglabplhak/pdfdb/20220501_DLBCSIA01_Reading_List.pdf
Number of pages: 6
Processing /Users/hglabplhak/pdfdb/DLMDSML01_6_Genetic_Algorithm_20260305.pdf
Number of pages: 48
Processing /Users/hglabplhak/pdfdb/General citation guidelines.pdf
Number of pages: 24
Processing /Users/hglabplhak/pdfdb/Gradient-based_Model_Shortcut_Detection_for_Time_S.pdf
Number of pages: 6
Processing /Users/hglabplhak/pdfdb/r6rs.pdf
Number of pages: 90
Processing /Users/hglabplhak/pdfdb/001-2023-0713_DLMDSDL01_Course_Book.pdf
Number of pages: 144
Processing /Users/hglabplhak/pdfdb/DLMDSML01 Mastersolution.pdf
Number of pages: 10
Processing /Users/hglabplhak/pdfdb/978-3-031-95365-1.pdf
Number of pages: 597
Processing /Users/hglabplhak/pdfdb/Systematic_Compiler_Construction.pdf
Number of pages: 13
Processing /Users/hglabplhak/pdfdb/Assembly language for x86 processors  Kip R. Irvine. 6th ed.pdf
Number of pages: 705
Processing /Users/hglabplhak/pdfdb/ex_guide_guidelines_for_online_presentations.pd

Ignoring wrong pointing object 113 0 (offset 0)
Ignoring wrong pointing object 114 0 (offset 0)
Ignoring wrong pointing object 115 0 (offset 0)
Ignoring wrong pointing object 116 0 (offset 0)
Ignoring wrong pointing object 117 0 (offset 0)
Ignoring wrong pointing object 118 0 (offset 0)
Ignoring wrong pointing object 119 0 (offset 0)
Ignoring wrong pointing object 120 0 (offset 0)
Ignoring wrong pointing object 121 0 (offset 0)
Ignoring wrong pointing object 123 0 (offset 0)
Ignoring wrong pointing object 124 0 (offset 0)
Ignoring wrong pointing object 125 0 (offset 0)
Ignoring wrong pointing object 126 0 (offset 0)
Ignoring wrong pointing object 127 0 (offset 0)
Ignoring wrong pointing object 128 0 (offset 0)
Ignoring wrong pointing object 129 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 131 0 (offset 0)
Ignoring wrong pointing object 132 0 (offset 0)
Ignoring wrong pointing object 133 0 (offset 0)
Ignoring wrong pointing object 134 0 (of

Processing /Users/hglabplhak/pdfdb/ELF - 238P Operating Systems.pdf
Number of pages: 1
Processing /Users/hglabplhak/pdfdb/DLMDSML01_5_Decision_Trees_20260303.pdf
Number of pages: 43


Ignoring wrong pointing object 704 0 (offset 0)
Ignoring wrong pointing object 707 0 (offset 0)
Ignoring wrong pointing object 710 0 (offset 0)
Ignoring wrong pointing object 713 0 (offset 0)
Ignoring wrong pointing object 718 0 (offset 0)
Ignoring wrong pointing object 723 0 (offset 0)
Ignoring wrong pointing object 728 0 (offset 0)
Ignoring wrong pointing object 731 0 (offset 0)
Ignoring wrong pointing object 734 0 (offset 0)
Ignoring wrong pointing object 737 0 (offset 0)
Ignoring wrong pointing object 740 0 (offset 0)
Ignoring wrong pointing object 745 0 (offset 0)
Ignoring wrong pointing object 748 0 (offset 0)
Ignoring wrong pointing object 749 0 (offset 0)
Ignoring wrong pointing object 752 0 (offset 0)
Ignoring wrong pointing object 759 0 (offset 0)
Ignoring wrong pointing object 764 0 (offset 0)
Ignoring wrong pointing object 769 0 (offset 0)
Ignoring wrong pointing object 774 0 (offset 0)
Ignoring wrong pointing object 779 0 (offset 0)
Ignoring wrong pointing object 782 0 (of

Processing /Users/hglabplhak/pdfdb/Executable and Linkable Format - Wikipedia.pdf
Number of pages: 2
Processing /Users/hglabplhak/pdfdb/DLMDSML01_Session 3.pdf
Number of pages: 31
Processing /Users/hglabplhak/pdfdb/MIT-LCS-TM-030.pdf
Number of pages: 119
